<!-- nav -->
[← 8 · Tools that work for every schema](08_Tools_For_Every_Schema.ipynb) · [Home](../../README.md) · [Schemas Framework design →](../../docs/FRAMEWORK.md)

# 9 · When requirements change: evolving schemas

**The problem.** The CRM has been in production for a year. Now product wants:

1. a `pronouns` field on contacts, without breaking the thousands of saved documents,
2. a v2 contact with a different shape, running side by side with v1 during migration,
3. customer IDs that are unique across the whole system, and
4. contacts reachable by phone *or* email: a variant, a union.

Schemas are values built in the store, so changing a schema is ordinary code, with the same calls you use for
instances: a schema and a function update it, and a builder started from it clones it. This last case study walks
through each request and ends with what the framework does *not* do yet.

In [1]:
import { Builders, Errors, JSON as Json, Proxies, Schemas, Validators } from "@mbse/schemas/Framework";

const store: any = new Builders.Store();
const Text = store.Schemas.OfStr;
const Int = store.Schemas.OfInt;

const { DecodeError } = Errors;

const { Contact, saved } = store.transaction(() => {
  const Contact = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Contact") // a namespace, then the schema's own name
      .ref(true)
      .properties([
        (t: any) => t.name("given_name").schema([Text]),
        (t: any) => t.name("family_name").schema([Text]),
      ])
  ));

  // A dotted name is reached one part at a time: store.crm.Contact.
  const saved = Json.ToJSON(store)(Contact, store.crm.Contact((c: any) => c.given_name("Alice").family_name("Liddell")));
  return { Contact, saved };
});
console.log(saved);

{"mutations": [{"upsert": {"exists": [{"symbols": {"0": {"schema": [{"$ref": {"singleton": ".", "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "crm"}, "next": {"entry": {"i": "schemas", "o": "", "properties": {"name": "Contact"}, "next": {"target": {}}}}}}}}]}}, "spec": [{"0": [{"as_property": {"name": "given_name", "mutations": [{"set": {"value": "Alice"}}]}}, {"as_property": {"name": "family_name", "mutations": [{"set": {"value": "Liddell"}}]}}]}]}]}}]}


## 1. Adding a field

Called with the schema and a function, `store.Schemas.OfObject(Contact, ...)` changes the schema *in place*. A program
that changes its schemas as it starts can also say what must hold of them rather than what to do: a transaction's
*expression*, here an `upsert` that finds the schema object by its name and writes what its spec gives where it does
not hold already. Run again, it finds the property there and changes nothing.

A schema that existed before a transaction changes when the transaction ends. From then on everything registered
under it sees the change, existing instances included; within the transaction that made it, the schema is as it was,
so data with the new field is written in the next one:

In [2]:
/** The change, as a transaction's expression: crm.Contact, found by its name, with a `pronouns` property. */
const pronouns = (t: any): any => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({ Contact: store.Schemas.OfObject }).spec([
  (_: any) => _.Contact.name("crm.Contact").properties({ pronouns: Text }),
])]))]);

store.transaction(pronouns);
store.transaction(pronouns); // run again: nothing to change

const alice = store.transaction(() => {
  const alice = Json.FromJSON(store)(Contact, saved) as Proxies.Instance; // a document saved before the change
  store.crm.Contact(alice, (c: any) => c.pronouns("she/her"));
  return alice;
});
console.log(`${alice.given_name}`, `${alice.pronouns}`);

Alice she/her


Old data loads without a migration step, because **every property is optional** (case study 1). A new property is
simply absent in old data. This is the payoff for not making fields mandatory by default.

The other direction is strict. A program still running the *old* schema, with a store of its own, refuses data
carrying the new field, rather than silently dropping it on the next save:

In [3]:
const oldStore: any = new Builders.Store();
const { OldContact } = oldStore.transaction(() => {
  const OldContact = oldStore.Schemas.OfObject((o: any) => ( // the schema as it was, under the same name, in a store of its own
    o
      .name("crm.Contact")
      .ref(true)
      .properties([
        (t: any) => t.name("given_name").schema([oldStore.Schemas.OfStr]),
        (t: any) => t.name("family_name").schema([oldStore.Schemas.OfStr]),
      ])
  ));
  return { OldContact };
});
const newer = Json.ToJSON(store)(Contact, alice);
try {
  Json.FromJSON(oldStore)(OldContact, newer);
} catch (error) {
  if (!(error instanceof DecodeError)) throw error;
  console.log(String(error));
}

DecodeError: $.mutations[0].upsert.exists[0].spec[0]["0"][2].as_property: crm.Contact has no property 'pronouns'


**Why refuse?** Loading is the last moment the program can notice that it doesn't understand the data. Dropping the
field would lose it on the next save, with nothing in any log. Deploy readers before writers, as with any strict
format.

## 2. A second version, side by side

A clone is an independent copy to change. In its store a schema's path is its name, so a clone of `Contact` takes a
new name: a builder in the store, started from `Contact`, renames the copy before `clone()` ends it. The builder's
arguments are the schema it starts from, whether it may write (`true`) and the store. The clone is registered under
its new name, in a module of its own, `crm.v2` (a module and a schema do not share a path), and v1 and v2 coexist:

In [4]:
const { ContactV2 } = store.transaction(() => {
  const ContactV2 = (
    new Builders.OfObject.Builder(Contact, true, store)
      .name("crm.v2.Contact")
      .properties([(t: any) => t.name("display_name").schema([Text])])
      .clone()
  );
  return { ContactV2 };
});
console.log(store.registered("crm.v2.Contact") === ContactV2);

console.log([...Contact.properties.keys()]);
console.log([...ContactV2.properties.keys()]);

true


[ "given_name", "family_name", "pronouns" ]


[ "given_name", "family_name", "pronouns", "display_name" ]


A migration is ordinary code over the two schemas:

In [5]:
function migrate(v1: Proxies.Instance): Proxies.Instance {
  return store.crm.v2.Contact((c: any) => (
    c
      .given_name(v1.given_name)
      .family_name(v1.family_name)
      .display_name(`${v1.given_name} ${v1.family_name}`)
  ));
}

const v2 = store.transaction(() => migrate(alice));
console.log(v2.as_schema().name, "|", `${v2.display_name}`);

crm.v2.Contact | Alice Liddell


Because references carry schema names (case study 6), a document that mixes v1 and v2 objects loads each with the
right schema.

Redefining a property replaces it, and that is how a property's schema changes. `validate()` on the result, and on your data,
tells you what else needs to follow.

## 3. Unique customer IDs: a directory

"Every customer has an ID, and an ID names at most one customer" sounds like it needs a special feature. It's a
relation with `unique` clauses (case study 5), linking a directory object to each customer, with the ID as the entry
property:

In [6]:
const { Customers, Directory } = store.transaction(() => {
  const Customers = store.Schemas.OfRelation((r: any) => (
    r
      .name("crm.Customers")
      .links([
        (t: any) => t.name("directory"),
        (t: any) => t.name("customer"),
      ])
      .properties([(t: any) => t.name("id").schema([Text])])
      .uniques([
        ["customer"], // (directory, id) determine the customer: an ID resolves to one customer
        ["id"], // (directory, customer) determine the id: a customer has one ID
      ])
  ));
  const Directory = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Directory")
      .ref(true)
      .singleton("crm.Directory") // declared as the one global directory
      .adjacencies([(t: any) => t.name("customers").relation([Customers]).me("directory")])
  ));
  store.Schemas.OfObject(ContactV2, (o: any) => o.adjacencies([
    (t: any) => t.name("ids").relation([Customers]).me("customer"),
  ]));
  return { Customers, Directory };
});

const directory = store.singleton("crm.Directory"); // made by the store, with its schema
const { bob } = store.transaction(() => {
  const bob = migrate(store.crm.Contact((c: any) => c.given_name("Bob").family_name("Kane")));
  store.crm.Directory(directory, (d: any) => d.customers((a: any) => a.insert((x: any) => x.customer(v2).id("C-1"))));
  store.crm.Directory(directory, (d: any) => d.customers((a: any) => a.insert((x: any) => x.customer(bob).id("C-1")))); // oops
  return { bob };
});

console.log(Validators.Validate(store).Reachable(Directory, directory));

[
  "unique(customer) violated: entries agreeing on ['directory', 'id'] differ on ['customer']"
]


Named references, maps, directories and ownership are all the same element: a relation plus `unique`. There's no
separate "ID" or "foreign key" feature to learn.

The `.singleton("crm.Directory")` declaration records that there's exactly one directory, referenced by its global
name. The store makes that instance when the schema is built and registered, and `store.singleton("crm.Directory")`
gives it; building another is refused. A store's singletons are its roots: the store keeps what they reach, here the
directory and every customer listed in it, and anything they don't reach is the program's to keep.

## 4. Variants: unions and intersections

"Reachable by phone or email" is a **union**: one of several same-kind schemas, each a named branch, and a value
holds exactly one of them. "Timestamped and audited" is an **intersection**: a value holding each of several named
parts. Each part keeps its own properties, so two parts may both declare `updated`, even with different schemas. The
branches and parts are lists of entries, each named and given a schema as a property is:

In [7]:
const { Phone, Email, Reach } = store.transaction(() => {
  const Phone = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("number").schema([Text])]));
  const Email = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("address").schema([Text])]));

  const Reach = store.Schemas.OfUnion((u: any) => u.branches([
    (t: any) => t.name("phone").schema([Phone]),
    (t: any) => t.name("email").schema([Email]),
  ]));
  return { Phone, Email, Reach };
});
console.log("union:", Reach.validate());

const { Timestamped, Epoch, Stamps } = store.transaction(() => {
  const Timestamped = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("updated").schema([Text])]));
  const Epoch = store.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("updated").schema([Int])]));
  const Stamps = store.Schemas.OfIntersection((i: any) => i.parts([
    (t: any) => t.name("text").schema([Timestamped]),
    (t: any) => t.name("epoch").schema([Epoch]),
  ]));
  return { Timestamped, Epoch, Stamps };
});
console.log("intersection:", Stamps.validate());

union: []


intersection: []


Unions and intersections can be declared and validated as schemas. A property whose schema is an object holds a
*value* object: a read-only value that belongs to its owner, is copied with it and is written nested in it. Union and intersection values
are written the same way, keyed by name: a union value `{"email": {"address": "..."}}` holds its one branch, and an
intersection value `{"text": {...}, "epoch": {...}}` each of its parts. Which branch a value holds is data, so
neither loading nor validation needs predicates, and reordering a union's branches leaves stored data valid.

A generated binding may store a union the way its language does (a tagged union, `std::variant`); the named form is
the neutral one that every program reads.

## What's designed but not built yet

The design document (`FRAMEWORK.md`) specifies more than the TypeScript package implements today. It's worth knowing
where the edges are:

| Feature | Status |
|---|---|
| Expressions (constraints such as "at least one") | in the separate [mbse-expressions](https://github.com/pitaman71/mbse-expressions) package, which depends on this one |
| Transactions as serializable data: a store's snapshot and journal made of them | built: a transaction is written as its expression, a document as one transaction, a store's snapshot as two, and a journal is a list of them |
| Deleting objects (entries linking them vanish with them) | an owned object goes with the entry that owns it; deleting any other is designed, not implemented |
| Generated bindings, with precise TypeScript types per schema | planned, in separate repositories per target language; the dynamic proxies are what exists |

Open questions are listed in `FRAMEWORK.md` under *Open questions*, several of them with a test that pins today's
behavior (for example F4, "last `update()` wins", which you met in case study 3).

## The whole tour

| Case study | Problem | Idea |
|---|---|---|
| 1 | a contact card | schemas are values; building in a store: create, update, clone; absent is not `undefined` |
| 2 | an address book | relations instead of arrays of reference objects; untyped links; entries are shared facts; containers of values |
| 3 | moving house | editing entries through builders; what an update replaces |
| 4 | a family tree | self-relations, cycles, `Reachable.of`; unambiguous inference |
| 5 | a price list | `unique(...)` cardinality; validation on demand |
| 6 | saving and loading | documents: one transaction, its objects symbols numbered from the root; paths; schema names beside the content |
| 7 | JSON and YAML | strict text; the Norway problem; no coercion |
| 8 | tools for every schema | walking schemas; reading any object through its visitor |
| 9 | evolving schemas | optional fields; a schema change as a transaction's expression; versions; directories; variants |

The through-line: **describe data once, as data, and let every tool read that description.** Most of the ways this
framework differs from the usual approach (no arrays of reference objects, no mandatory fields, no automatic
validation, no coercion, no guessing) are there so that description means the same thing to every program that
reads it, in TypeScript, in Python, and in whatever comes next.

<!-- nav -->
[← 8 · Tools that work for every schema](08_Tools_For_Every_Schema.ipynb) · [Home](../../README.md) · [Schemas Framework design →](../../docs/FRAMEWORK.md)